<a href="https://colab.research.google.com/github/RAPID-Facility/rAPIdtools/blob/main/examples/asset_analysis_example_generalized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Automated Extraction and AI Analysis of Infrastructure Assets from Multi-Source Imagery Using `rAPIdtools`**

In [1]:
!pip install rapidtools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.5/210.5 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 67.6 MB/s eta 0:00:00
  Attempting uninstall: pyshp
    Found existing installation: pyshp 3.1.4
    Uninstalling pyshp-3.1.4:
      Successfully uninstalled pyshp-3.1.4
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.1

## <u>Part 1: Automated Extraction of Assets from RAPID Imagery</u>
In this first step, we automatically identify and map all buildings within the disaster zone without dependence on pre-existing GIS databases. By integrating high-resolution aerial imagery with Meta's Segment Anything Model 3 (SAM 3), we construct a foundational asset inventory from the ground up. Although this example emphasizes buildings, the process is applicable to any physical asset discernible in aerial imagery.

### Define the assets you would like to detect & Define Gemini API Key

In [12]:
TARGET_ASSET = 'tree' # free-form text
APPRX_ASSET_SIZE=50 # in meters
DETECT_IN_RECON_IMAGERY= False # Set to True if the asset can only be detected in post-disaster imagery

API_KEY = 'ENTER-API-KEY-HERE'

### Import required rAPIdtools methods and packages

In [4]:
from rapidtools import (
    AerialImageryExtractor,
    BingOrthomosaicExtractor,
    BoundingBox,
    BuildingRegularizer,
    GeminiAssetAnalyzer,
    MapillaryImageExtractor,
    SAM3OrthoFeatureExtractor,
    Pipeline,
    PolygonRegion,
    PhysicalAssetCollection,
    RoadwayRegularizer,
    download_dataset
)

from pathlib import Path

2026-07-22 18:39:56- INFO: NumExpr defaulting to 2 threads.
2026-07-22 18:40:14- INFO: Preparing to download 1 file(s) across 1 dataset(s)...


2026-07-22 18:40:14- INFO: Successfully secured 1/1 files.


2026-07-22 18:40:15- INFO: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-07-22 18:40:15- INFO: HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
2026-07-22 18:40:15- INFO: Hugging Face auto-authentication successful.


### Download a UW RAPID post-event raster for the 2025 Eaton Fire in Altadena, CA and determine its bounds


In [5]:
[raster_path] = download_dataset('eaton_patch1')
region_of_interest = BoundingBox.from_raster(raster_path)

2026-07-22 18:40:19- INFO: Preparing to download 1 file(s) across 1 dataset(s)...


2026-07-22 18:42:48- INFO: Successfully secured 1/1 files.


### Extract preliminary asset geometries using Meta's Segment Anything Model (SAM 3)

In [6]:
sam_extractor = SAM3OrthoFeatureExtractor(
    prompt=TARGET_ASSET,
    patch_size=APPRX_ASSET_SIZE,
    unit='meters',
    overlap_ratio=0.25,
    batch_size=4,
    threshold=0.50,
    mask_threshold=0.40,
)

if DETECT_IN_RECON_IMAGERY:
    prelim_collection = sam_extractor(raster_path)
else:
  bing_extractor = BingOrthomosaicExtractor(max_workers=10)
  bing_raster_path = bing_extractor(
      region=region_of_interest,
      output_path='eaton_patch1_bing.tiff'
  )
  prelim_collection = sam_extractor(bing_raster_path)

_ = prelim_collection.to_geojson('eaton_patch1_assets_preliminary.geojson')

2026-07-22 18:44:32- INFO: Initializing SAM3OrthoFeatureExtractor for 'tree' at scale: 50 meters with threshold 0.5...
2026-07-22 18:44:32- INFO: Loading processor and weights for facebook/sam3 (this may take a minute)...


processor_config.json:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

2026-07-22 18:45:22- INFO: SAM 3 model facebook/sam3 loaded successfully.
2026-07-22 18:45:22- INFO: Stitching 1991x1355 px synthetic orthomosaic at zoom 19...
2026-07-22 18:45:23- INFO: Saving GeoTIFF to: /content/eaton_patch1_bing.tiff


Scanning for 'tree': 0it [00:00, ?it/s]

2026-07-22 18:45:27- WARNING: CPLE_AppDefined in DeprecationWarning: 'Memory' driver is deprecated since GDAL 3.11. Use 'MEM' onwards. Further messages of this type will be suppressed.


Scanning for 'tree': 126it [01:03,  1.98it/s]


2026-07-22 18:46:28- INFO: Extracted 482 raw polygons. Merging overlaps...
2026-07-22 18:46:28- INFO: Extraction complete. Yielded 270 unique assets.


### Extract buffered image crops to provide spatial context for geometric regularization


In [7]:
extractor = AerialImageryExtractor(
    dataset=bing_raster_path,
    save_directory='output/asset_crops',
    buffer_asset='20 m',
    force_square_image=True,
)

assets_with_images = extractor(prelim_collection)

2026-07-22 18:47:50- INFO: Found 1 raster(s). Images will be saved to: /content/output/asset_crops
2026-07-22 18:47:50- INFO: Processing raster: eaton_patch1_bing.tiff
2026-07-22 18:47:50- INFO: Found 270/270 assets within the current raster's extent.


Extracting from eaton_patch1_bing.tiff: 100%|██████████| 270/270 [00:02<00:00, 125.49it/s]

2026-07-22 18:47:53- INFO: Finished processing all rasters. Extracted aerial imagery for a total of 270 assets.


### Refine and regularize asset geometries into clean, GIS-ready polygons


In [9]:
if 'building' in TARGET_ASSET:
    regularizer = BuildingRegularizer(batch_size=8)
elif 'road' in TARGET_ASSET:
    regularizer = RoadwayRegularizer()
else:
  regularizer = None

if regularizer:
  final_assets = regularizer(assets_with_images)
else:
  final_assets = assets_with_images

_ = final_assets.to_geojson('eaton_patch1_assets_final.geojson')

## <u>Part 2: Aerial Post-Disaster Condition Assessment</u>

### Configure the extraction and analysis pipeline

In [13]:
[prompt_path] = download_dataset('aerial_chs_prompts')
damage_classification_prompt = Path(prompt_path).read_text().strip()

pipeline = Pipeline()

extractor = AerialImageryExtractor(
    dataset=raster_path,
    save_directory='eaton_fire_aerial_feb25/overlaid_imagery',
    overlay_asset_outline=True,
    image_prefix='eaton_trinity_25',
    keep_multiple_copies=True,
)

analyzer = GeminiAssetAnalyzer(
    api_key=API_KEY,
    prompt=damage_classification_prompt,
)

2026-07-22 18:51:43- INFO: Preparing to download 1 file(s) across 1 dataset(s)...
2026-07-22 18:51:43- INFO: File already exists at: /content/aerial_CHS_prompts.txt. Skipping.
2026-07-22 18:51:43- INFO: Successfully secured 1/1 files.


### Execute the workflow and export results

In [14]:
pipeline.add_step(extractor)
pipeline.add_step(analyzer)

processed_collection = pipeline.run(final_assets)

final_collection = processed_collection.filter_empty()

_ = final_collection.to_geojson(
    'eaton_assets_inferred.geojson',
    ignore_properties=['image_assets']
)

2026-07-22 18:51:46- INFO: Starting pipeline with 2 steps...
2026-07-22 18:51:46- INFO: --- Running step 1/2: AerialImageryExtractor ---
2026-07-22 18:51:46- INFO: Found 1 raster(s). Images will be saved to: /content/eaton_fire_aerial_feb25/overlaid_imagery
2026-07-22 18:51:46- INFO: Processing raster: eaton_patch_20250214.tiff
2026-07-22 18:51:47- INFO: Found 270/270 assets within the current raster's extent.


Extracting from eaton_patch_20250214.tiff: 100%|██████████| 270/270 [00:16<00:00, 16.14it/s]

2026-07-22 18:52:03- INFO: Finished processing all rasters. Extracted aerial imagery for a total of 270 assets.
2026-07-22 18:52:03- INFO: --- Running step 2/2: GeminiAssetAnalyzer ---
2026-07-22 18:52:03- INFO: Gemini API: Analyzing 270 assets with 5 workers (Base Cooldown: 30s).



Analyzing Assets:  84%|████████▍ | 228/270 [01:12<00:13,  3.11it/s]

2026-07-22 18:53:16- ERROR: Rate limit or API error hit on tree_db293c95. Consecutive errors: 7. Global cooldown set for 210s.
2026-07-22 18:53:16- INFO: Asset tree_e56d8b42 delayed: Global cooldown active for 209s...


Analyzing Assets:  85%|████████▍ | 229/270 [01:13<00:14,  2.92it/s]

2026-07-22 18:53:17- ERROR: [Prompt snippet: 'TASK: Analyze each provided im...'] HTTP Error: 403 Client Error: Forbidden for url: https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent | Response: {
  "error": {
    "code": 403,
    "message": "Your API key was reported as leaked. Please use another API key.",
    "status": "PERMISSION_DENIED"
  }
}

2026-07-22 18:53:17- ERROR: [Prompt snippet: 'TASK: Analyze each provided im...'] HTTP Error: 403 Client Error: Forbidden for url: https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent | Response: {
  "error": {
    "code": 403,
    "message": "Your API key was reported as leaked. Please use another API key.",
    "status": "PERMISSION_DENIED"
  }
}

2026-07-22 18:53:17- ERROR: [Prompt snippet: 'TASK: Analyze each provided im...'] HTTP Error: 403 Client Error: Forbidden for url: https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-

Analyzing Assets: 100%|██████████| 270/270 [01:26<00:00,  3.13it/s]

2026-07-22 18:53:30- INFO: Gemini API: All assets processed successfully.
2026-07-22 18:53:30- INFO: Pipeline execution successfully completed.
